# 09 · Fine-tuning Encoder dengan Objektif Pemeringkatan (Opsional)
Lihat `laporan/arsip/2026-11-07_sebelum_run/catatan/kritik-reviewer.md` Bagian 3.3. Encoder **tidak dibekukan** dan dilatih bersama kepala MLP dengan loss RankNet di dalam produk.
Agar adil, BLaIR dan RoBERTa-base sama-sama di-*fine-tune*, lalu dibandingkan dengan versi beku (U dan B5) pada skala dan data yang sama.

**Biaya:** jauh lebih mahal dari model beku karena setiap langkah menjalankan seluruh Transformer. Karena itu dipakai skala E1,
panjang 256 token, dan subset produk validasi untuk *early stopping*. Perlu GPU minimal 16 GB.

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

In [ ]:
import torch
from hr import experiment as X, finetune, features, report as R
b = X.load_bundle(cfg)
best_hp = json.loads(cfg.path("best_hparams.json").read_text())
SC = list(cfg.scales)[0]                         # E1
tr, te = b.rows("train", SC), b.rows("test")
va_all = b.rows("val")
keep = va_all["parent_asin"].drop_duplicates().sample(min(1000, va_all["parent_asin"].nunique()), random_state=0)
va = va_all[va_all["parent_asin"].isin(keep)]
Z = {n: features.struct_matrix(d, "all").astype(np.float32) for n, d in (("tr", tr), ("va", va), ("te", te))}
print(SC, len(tr), "ulasan latih |", len(va), "ulasan validasi (subset) | GPU:", torch.cuda.is_available())

## 1. Pembanding beku pada skala yang sama

In [ ]:
for code in ["U", "B5"]:
    for seed in cfg.seeds[:1]:
        X.run_torch(b, code, scale=SC, target="T1", seed=seed, **best_hp, verbose=False)

## 2. Fine-tuning BLaIR dan RoBERTa-base

In [ ]:
FT_LEN = 256 if not SMOKE else 64
for name in ["blair", "roberta"]:
    run = f"FT{name}_{SC}_T1_s{cfg.seed}"
    out = cfg.out_dir / "runs" / run
    if (out / "result.json").exists():
        continue
    model, tok, hist, best = finetune.finetune(
        cfg.encoders[name], tr, tr["y_t1"].to_numpy(), Z["tr"], va, Z["va"], va["grade_t1"].to_numpy(),
        va["y_t1"].to_numpy(), max_len=FT_LEN, epochs=2 if not SMOKE else 1,
        eval_every=1000 if not SMOKE else 5, seed=cfg.seed, map_rel=cfg.map_relevant_grade)
    dev = next(model.parameters()).device.type
    out.mkdir(parents=True, exist_ok=True)
    hist.to_csv(out / "history.csv", index=False)
    s_va = finetune.score_frame(model, tok, va_all, features.struct_matrix(va_all, "all").astype(np.float32), FT_LEN, dev)
    s_te = finetune.score_frame(model, tok, te, Z["te"], FT_LEN, dev)
    X.save_scores(cfg, run, {"code": f"FT-{name}", "scale": SC, "target": "T1", "seed": cfg.seed,
                             "best_val_ndcg@10": best, "max_len": FT_LEN}, va_all, s_va, te, s_te)
    del model; torch.cuda.empty_cache()

## 3. Perbandingan pada data uji
Dijalankan setelah notebook 07 agar keputusan utama tidak terpengaruh.

In [ ]:
groups = R.group_runs(X.results_table(cfg))
keys = [("U", SC, "T1"), ("B5", SC, "T1"), ("FT-blair", SC, "T1"), ("FT-roberta", SC, "T1")]
ft_tab, ft_pps = R.comparison_table(b, groups, keys, "T1", cfg.eval_bootstrap)
ft_tab.to_csv(cfg.path("tables", "uji_finetuning.csv"), index=False)
display(ft_tab[["code", "seeds", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi", "spearman", "map"]].round(4))
R.paired_table(ft_pps, ("FT-blair", SC, "T1"), [k for k in keys if k[0] != "FT-blair"]).round(4)

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()